# Fake News Detection Using a Neural Network

This notebook classifies news headlines into:
- 0 = Real News
- 1 = Fake News

In [ ]:
# Uncomment in Colab if needed
# !pip install tensorflow scikit-learn pandas numpy matplotlib

import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Step 1: Load and Explore Dataset

In [ ]:
DATASET_PATH = '/content/fake_news_dataset.csv'  # Update this path

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

df = pd.read_csv(DATASET_PATH)
required_cols = {'text', 'label'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

print('Dataset shape:', df.shape)
print(df.head())
print('\nClass distribution:')
print(df['label'].value_counts().sort_index())

## Step 2: Text Preprocessing (TF-IDF)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = df[['text', 'label']].dropna().copy()
df['text'] = df['text'].apply(clean_text)

X_text = df['text'].values
y = df['label'].astype(int).values

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(X_text).toarray()
print('Feature shape:', X.shape)

## Step 3: Split Dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print('Train:', X_train.shape, y_train.shape)
print('Test :', X_test.shape, y_test.shape)

## Step 4: Build Neural Network

In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 5: Train Model

In [ ]:
EPOCHS = 12
BATCH_SIZE = 32

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.show()

## Step 6: Evaluate Model

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

## Step 7: Manual Testing

In [ ]:
def predict_headline(headline):
    clean = clean_text(headline)
    vec = vectorizer.transform([clean]).toarray()
    prob_fake = float(model.predict(vec, verbose=0)[0][0])
    label = 'Fake News' if prob_fake >= 0.5 else 'Real News'
    return label, prob_fake

examples = [
    'Scientists confirm water on Mars.',
    'Secret government project creates invisible humans.'
]

for text in examples:
    label, prob = predict_headline(text)
    print(f'Headline: {text}')
    print(f'Prediction: {label}, fake_probability={prob:.4f}\n')

## Save Trained Model

In [ ]:
MODEL_PATH = 'fake_news_nn_model.keras'
VECTORIZER_PATH = 'tfidf_vectorizer.pkl'

model.save(MODEL_PATH)
with open(VECTORIZER_PATH, 'wb') as f:
    pickle.dump(vectorizer, f)

print('Saved:', MODEL_PATH)
print('Saved:', VECTORIZER_PATH)